## Setup and Imports

In [2]:
import json
import pandas as pd
import os

# Define file paths (adjust these if your files are in a specific subdirectory)
json_files = [
    'barchart_horizontal.json', 
    'barchart_vertical.json', 
    'line_chart.json', 
    'pie_chart.json', 
    'scatter_plot.json'
]

csv_files = {
    'train': 'train.csv',
    'val': 'val.csv',
    'test1': 'test.csv',
    'test2': 'test_two.csv'
}

## Parse JSON Metadata (Titles and Chart Types)

In [10]:
import urllib.parse

image_metadata = {}

def extract_metadata(data, chart_type):
    """Recursively searches for image records in any nested JSON structure."""
    if isinstance(data, dict):
        # Look for the 'file_name' key based on the actual ChartCheck schema
        raw_id = data.get('file_name', data.get('image_name', data.get('id', '')))
        
        if raw_id:
            # Decode URL-encoded characters (e.g., %20 to spaces) for perfect matching
            img_id = urllib.parse.unquote(raw_id)
            description = data.get('description', data.get('title', ''))
            
            image_metadata[img_id] = {
                'description': description,
                'chart_type': chart_type
            }
        else:
            for value in data.values():
                extract_metadata(value, chart_type)
                
    elif isinstance(data, list):
        for item in data:
            extract_metadata(item, chart_type)

# Process each file
for file in json_files:
    chart_type = file.replace('.json', '')
    
    try:
        with open(file, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
            extract_metadata(raw_data, chart_type)
    except FileNotFoundError:
        print(f"Warning: {file} not found. Skipping.")
    except Exception as e:
        print(f"Error reading {file}: {e}")

print(f"Successfully loaded metadata for {len(image_metadata)} unique images.")

Successfully loaded metadata for 2494 unique images.


## Load the Claim Data Splits

In [11]:
data_splits = {}

for split_name, file_path in csv_files.items():
    if os.path.exists(file_path):
        data_splits[split_name] = pd.read_csv(file_path)
        print(f"Loaded {split_name}: {len(data_splits[split_name])} claims.")
    else:
        print(f"Warning: {file_path} not found.")

train_df = data_splits.get('train', pd.DataFrame())
val_df = data_splits.get('val', pd.DataFrame())
test1_df = data_splits.get('test1', pd.DataFrame())
test2_df = data_splits.get('test2', pd.DataFrame())

Loaded train: 7607 claims.
Loaded val: 953 claims.
Loaded test1: 939 claims.
Loaded test2: 981 claims.


## Download Images Locally (Multithreaded)

In [15]:
import os
import requests
import urllib.parse
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Create a directory to store the images
IMG_DIR = "local_images"
os.makedirs(IMG_DIR, exist_ok=True)

# 1. Gather all unique image URLs from the CSV files
all_urls = set()
for df in [train_df, val_df, test1_df, test2_df]:
    if not df.empty:
        img_col = 'chart_img' if 'chart_img' in df.columns else 'image_name'
        all_urls.update(df[img_col].dropna().astype(str).tolist())

def download_image(url):
    if not url.startswith('http'):
        url = 'https:' + url if url.startswith('//') else url
        
    # Extract the decoded filename to match your metadata dictionary
    raw_filename = url.split('/')[-1]
    filename = urllib.parse.unquote(raw_filename)
    save_path = os.path.join(IMG_DIR, filename)
    
    # Skip if already downloaded
    if os.path.exists(save_path):
        return True
        
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            return True
        else:
            print(f"Failed to download {url} - Status: {response.status_code}")
            return False
    except Exception as e:
        print(f"Error downloading {url}: {e}")
        return False

# 2. Download concurrently using 10 threads
print(f"Starting download of {len(all_urls)} images...")
success_count = 0

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = [executor.submit(download_image, url) for url in all_urls]
    
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading"):
        if future.result():
            success_count += 1

print(f"Done! Successfully downloaded {success_count}/{len(all_urls)} images to ./{IMG_DIR}/")

Starting download of 1684 images...


Downloading: 100%|██████████| 1684/1684 [00:00<00:00, 38823.13it/s]

Done! Successfully downloaded 1684/1684 images to ./local_images/


## Export Pre-Inference JSON Datasets

In [ ]:
import json
import urllib.parse
import os

# The directory where you downloaded the images in the previous step
IMG_DIR = "local_images"

def prepare_dataset_without_spec(df, metadata_dict):
    """
    Transforms the raw DataFrame into a structured list of dictionaries
    with local image paths, chart types, and necessary claim features.
    """
    dataset = []
    if df.empty:
        return dataset
        
    # Handle column name variations in the CSVs
    img_col = 'chart_img' if 'chart_img' in df.columns else 'image_name'
    
    for _, row in df.iterrows():
        raw_url = str(row.get(img_col, ''))
        
        # Extract just the filename from the end of the URL and decode it
        raw_filename = raw_url.split('/')[-1]
        img_id = urllib.parse.unquote(raw_filename)
        
        # Build the new local path mapped to your downloaded images
        local_path = os.path.join(IMG_DIR, img_id)
        
        # Fetch the metadata (Chart type and exact description/title)
        meta = metadata_dict.get(img_id, {})
        
        # Extract reasoning type if available in the CSV, otherwise default to 'unknown'
        reasoning_type = row.get('reasoning_type', row.get('type', 'unknown'))
        
        # Convert label to a standard boolean or string if needed
        label = row.get('label', row.get('label_claim', -1))
        if isinstance(label, str):
            label = label.strip().upper()
        
        dataset.append({
            'local_image_path': local_path,
            'original_url': raw_url,
            'claim': row.get('claim', ''),
            'label': label,
            'explanation': row.get('explanation', ''),
            'chart_type': meta.get('chart_type', 'unknown'),
            'reasoning_type': reasoning_type,
            'title_description': meta.get('description', row.get('caption', ''))
        })
        
    return dataset

# 1. Structure the data
train_data = prepare_dataset_without_spec(train_df, image_metadata)
val_data = prepare_dataset_without_spec(val_df, image_metadata)
test1_data = prepare_dataset_without_spec(test1_df, image_metadata)
test2_data = prepare_dataset_without_spec(test2_df, image_metadata)

# 2. Export to JSON files
with open('train_wo_spec.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, indent=4)
    
with open('validation_wo_spec.json', 'w', encoding='utf-8') as f:
    json.dump(val_data, f, indent=4)
    
with open('test_1_wo_spec.json', 'w', encoding='utf-8') as f:
    json.dump(test1_data, f, indent=4)
    
with open('test_2_wo_spec.json', 'w', encoding='utf-8') as f:
    json.dump(test2_data, f, indent=4)

print(f"Export Complete!")
print(f"Train rows: {len(train_data)}")
print(f"Validation rows: {len(val_data)}")
print(f"Test 1 rows: {len(test1_data)}")
print(f"Test 2 rows: {len(test2_data)}")

Export Complete!
Train rows: 7607
Validation rows: 953
Test 1 rows: 939
Test 2 rows: 981


## Inject Ground-Truth Reasoning Types

In [17]:
import json

# 1. Load the newly found reasoning JSONs
with open('claim_explanation_test_one_ID_reasoning.json', 'r', encoding='utf-8') as f:
    test1_reasoning = json.load(f)
    
with open('claim_explanation_test_two_ID_reasoning.json', 'r', encoding='utf-8') as f:
    test2_reasoning = json.load(f)

# 2. Create a lookup dictionary: (image_url, claim_text) -> reasoning_types list
reasoning_lookup = {}
for item in test1_reasoning + test2_reasoning:
    # The URL in these JSONs matches the original_url in our generated dataset
    img_url = item.get('chart_img', '')
    claim = item.get('claim', '')
    r_types = item.get('reasoning_types', ['unknown'])
    
    # Use a tuple of (url, claim) as a unique key since multiple claims exist per image
    reasoning_lookup[(img_url, claim)] = r_types

# 3. Function to update our existing pre-inference JSONs
def inject_reasoning_types(filename):
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
        updated_count = 0
        for row in data:
            key = (row.get('original_url', ''), row.get('claim', ''))
            if key in reasoning_lookup:
                # Replace the placeholder with the actual list of reasoning types
                row['reasoning_type'] = reasoning_lookup[key]
                updated_count += 1
                
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=4)
            
        print(f"Success: Updated {updated_count}/{len(data)} rows in {filename} with exact reasoning types.")
    except FileNotFoundError:
        print(f"Warning: {filename} not found in the current directory.")

# 4. Run the injection on your generated test files
inject_reasoning_types('test_1_wo_spec.json')
inject_reasoning_types('test_2_wo_spec.json')

Success: Updated 106/939 rows in test_1_wo_spec.json with exact reasoning types.
Success: Updated 67/981 rows in test_2_wo_spec.json with exact reasoning types.
